# Person 1 (Aarya): run the pipelines on a Colab GPU

**Before running anything:** menu **Runtime → Change runtime type → Hardware accelerator: GPU** (T4 is the free one; L4/A100 if you have Colab Pro) → Save.

Then run the cells **from top to bottom** (click a cell, press **Shift + Enter**). Each cell says what it does and what you should see.

If Colab disconnects, just run all cells again from the top: finished questions are skipped, nothing is lost *as long as you saved results* (step 7).

## 1. Check the GPU
You should see a table with a GPU name (Tesla T4, L4 or A100). If it says *command not found*, the runtime has no GPU: redo the menu step above.

In [6]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Download the project from GitHub
Re-running this cell later just pulls the newest version of `main`.

In [7]:
import os
REPO = "/content/LateComers-CSE576-NLP"
if not os.path.exists(REPO):
    !git clone https://github.com/aaryapendharkar/LateComers-CSE576-NLP.git {REPO}
%cd {REPO}
!git pull -q origin main
!git log --oneline -1

/content/LateComers-CSE576-NLP
155009c (HEAD -> main, origin/main, origin/HEAD) Merge pull request #5 from aaryapendharkar/p1-pipelines


## 3. Install the packages (2–4 minutes)
Ignore red *dependency conflict* warnings unless the cell ends with an error.

In [8]:
!pip install -q -r requirements-core.txt
!pip install -q -U transformers accelerate sentence-transformers
import transformers; print("transformers", transformers.__version__)

transformers 5.23.0


## 4. Pick the model that fits this GPU
Writes `configs/local.yaml` (only on this Colab machine; never uploaded).

The planned Qwen3.5-9B is not usable here: its new layer type needs extra GPU kernels, and without them **one question took ~5 minutes** on a T4 (Nov 13 test). We use **Qwen3-VL**, the same Qwen family with standard layers:
- T4 / L4 (≤ 30 GB): **Qwen/Qwen3-VL-4B-Instruct**
- A100 (Colab Pro): **Qwen/Qwen3-VL-8B-Instruct**

This is recorded in the report under *Changes*.

In [9]:
import os, torch, yaml
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory fragmentation
gpu = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
big = mem_gb >= 30
model = "Qwen/Qwen3-VL-8B-Instruct" if big else "Qwen/Qwen3-VL-4B-Instruct"
local = {
    "pipelines": {"backend": "real", "p1_text_source": "standin"},
    "models": {
        "generator_fallback": model,
        "reranker": None,
        "device": "cuda",
        "dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16",
        "batch_size": 8 if big else 4,
    },
}
with open("configs/local.yaml", "w") as f:
    yaml.safe_dump(local, f)
print(f"GPU: {gpu} ({mem_gb:.0f} GB)  ->  generator: {model}, dtype: {local['models']['dtype']}")

GPU: Tesla T4 (16 GB)  ->  generator: Qwen/Qwen3-VL-4B-Instruct, dtype: bfloat16


## 5. The model test: 5 questions through P1 (text pipeline)
First run downloads the models (BGE-M3 ≈ 2 GB, Qwen3-VL-4B ≈ 9 GB) and embeds all 500 pages once, so it takes **5–10 minutes**. After that, each question should take **seconds, not minutes**.

In [10]:
!rm -f results/runs/P1_R2_finqa.jsonl   # start the test fresh
!python scripts/run_baselines.py --pipelines P1 --runs R2 --limit 5

P1 R2:   0% 0/5 [00:00<?, ?it/s]
Loading weights: 100% 391/391 [00:00<00:00, 28312.49it/s]

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.32MB/s]

chat_template.json: 100% 5.50k/5.50k [00:00<00:00, 16.2MB/s]

config.json: 100% 1.50k/1.50k [00:00<00:00, 5.24MB/s]

tokenizer_config.json: 100% 10.9k/10.9k [00:00<00:00, 26.3MB/s]

vocab.json: 100% 2.78M/2.78M [00:00<00:00, 87.3MB/s]

merges.txt: 100% 1.67M/1.67M [00:00<00:00, 77.9MB/s]

tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 109MB/s]

video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 1.38MB/s]

model.safetensors.index.json: 100% 64.7k/64.7k [00:00<00:00, 118MB/s]


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            


Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]

Reconstructing (incomplete total...):   0% 0.00/8.88G [00:00<?, ?B/s]         

Reconstructing (incomplete total...):   0% 0.00/8.88G [00:00<?, ?B/s]

Reconstructing (incomplete total...):   0% 9.61k/8.88G [00:02<668:13:58, 3.69k

## 6. Look at the answers
**What good looks like:**
- `answer` is a number (or yes/no) for every row, not empty
- `cited` lists page ids
- `gold_rank` is mostly 1–5 (retrieval found the right page)
- `correct` is True for at least some rows (a few wrong answers are normal)

Copy this table into the chat with Claude.

In [11]:
import pandas as pd, json
rows = [json.loads(l) for l in open("results/runs/P1_R2_finqa.jsonl")]
ex = {json.loads(l)["qid"]: json.loads(l) for l in open("data/examples.jsonl")}
df = pd.DataFrame([{
    "qid": r["qid"], "gold": ex[r["qid"]]["gold_answer"][:10], "answer": r["answer"],
    "correct": r["correct"], "gold_rank": r["gold_rank_retr"], "cited": r["cited"],
    "secs": r["latency_s"], "raw_output_start": r["answer_raw"][:80],
} for r in rows])
pd.set_option("display.max_colwidth", 80)
df

,qid,gold,answer,correct,gold_rank,cited,secs,raw_output_start
0,q0001,-0.0291845,-2.9,True,NaN,[finqa_FIS_2016_p31__clean],182.527,"[shrunk x2 after OOM] {\n ""answer"": -2.9,\n ""cited_pages"": [\n ""finqa_F..."
1,q0002,0.08083333,8.0833,True,1.0,[finqa_MRO_2017_p96__clean],17.952,"[shrunk x2 after OOM] {\n ""answer"": 8.0833,\n ""cited_pages"": [\n ""finqa..."
2,q0003,0.70203359,69.9,True,NaN,[finqa_IPG_2015_p37__clean],17.454,"[shrunk x2 after OOM] {\n ""answer"": 69.9,\n ""cited_pages"": [\n ""finqa_I..."
3,q0004,0.35460122,36.5,False,1.0,[finqa_AMT_2007_p29__clean],17.300,"[shrunk x2 after OOM] {\n ""answer"": 36.5,\n ""cited_pages"": [\n ""finqa_A..."
4,q0005,2.67174959,2.67,True,1.0,[finqa_AAL_2013_p172__clean],18.294,"[shrunk x2 after OOM] {\n ""answer"": 2.67,\n ""cited_pages"": [""finqa_AAL_201..."


## 7. Save results to your Google Drive
Colab deletes everything when it disconnects. This copies `results/` to **My Drive / cse576_results** (it will ask permission to access Drive the first time).

In [12]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/cse576_results
!cp -r results/. /content/drive/MyDrive/cse576_results/
!ls /content/drive/MyDrive/cse576_results/runs

Mounted at /content/drive
P1_R2_finqa.jsonl


In [13]:
import yaml
cfg = yaml.safe_load(open("configs/local.yaml"))
cfg["generation"] = {"max_page_chars": 2000, "max_image_pixels": 300000}
yaml.safe_dump(cfg, open("configs/local.yaml", "w"))
print(open("configs/local.yaml").read())

generation:
  max_image_pixels: 300000
  max_page_chars: 2000
models:
  batch_size: 4
  device: cuda
  dtype: bfloat16
  generator_fallback: Qwen/Qwen3-VL-4B-Instruct
  reranker: null
pipelines:
  backend: real
  p1_text_source: standin



---
## 8. FULL P1 baselines: 200 questions × R1, R2, R3 (≈ 1–2 hours)
Run this only after step 6 looks good. It uses the page **text** straight from `data/docs.jsonl`, so it does not need Aaron's rendered pages. (When his pages arrive, we re-run P1 on the real PDF text and run P2 on the images.)

**If Colab disconnects:** run cells 1–4, 7 (mount Drive) and the *restore* cell below, then this cell again; finished questions are skipped.

In [14]:
# restore earlier progress from Drive (safe to run even the first time)
import os
if os.path.exists("/content/drive/MyDrive/cse576_results/runs"):
    !mkdir -p results/runs && cp -n /content/drive/MyDrive/cse576_results/runs/*.jsonl results/runs/ 2>/dev/null; ls results/runs

P1_R2_finqa.jsonl


In [15]:
!rm -f results/runs/P1_R2_finqa.jsonl   # drop the 5-question test so R2 is one clean run
!python scripts/run_baselines.py --pipelines P1 --runs R1 R2 R3
!mkdir -p /content/drive/MyDrive/cse576_results && cp -r results/. /content/drive/MyDrive/cse576_results/
!cat results/baselines/table1_finqa.md

P1 R1:   0% 0/200 [00:00<?, ?it/s]
Loading weights:   0% 0/713 [00:00<?, ?it/s]
Loading weights:   0% 1/713 [00:02<34:32,  2.91s/it]
Loading weights:   0% 3/713 [00:03<09:48,  1.21it/s]
Loading weights:   1% 4/713 [00:03<07:13,  1.63it/s]
Loading weights:   1% 5/713 [00:03<05:35,  2.11it/s]
Loading weights:   1% 9/713 [00:03<02:07,  5.53it/s]
Loading weights:   2% 11/713 [00:03<01:38,  7.12it/s]
Loading weights:   2% 14/713 [00:03<01:16,  9.08it/s]
Loading weights:   2% 16/713 [00:04<01:34,  7.39it/s]
Loading weights:   3% 20/713 [00:04<01:00, 11.40it/s]
Loading weights:   3% 23/713 [00:04<00:48, 14.18it/s]
Loading weights:   4% 26/713 [00:04<00:44, 15.59it/s]
Loading weights:   6% 44/713 [00:04<00:14, 46.33it/s]
Loading weights:   8% 59/713 [00:04<00:09, 67.25it/s]
Loading weights:  11% 75/713 [00:04<00:07, 88.33it/s]
Loading weights:  13% 92/713 [00:05<00:05, 104.46it/s]
Loading weights:  15% 110/713 [00:05<00:04, 121.29it/s]
Loading weights:  17% 124/713 [00:06<00:19, 30.00it/s] 
Lo

## 9. Download the results to upload to GitHub
Downloads `p1_results.zip`. Unzip it **into your LateComers-CSE576-NLP folder** on your laptop (choose *Replace*), then commit and push from PowerShell as usual.

In [1]:
!rm -f p1_results.zip && zip -qr p1_results.zip results/runs results/r2_correct.json results/baselines
from google.colab import files
files.download("p1_results.zip")


zip error: Nothing to do! (try: zip -qr p1_results.zip . -i results/runs results/r2_correct.json results/baselines)


FileNotFoundError: Cannot find file: p1_results.zip

---
## Later steps (don't run yet)
After Aaron's renderer and pages are on `main`, Claude will give you the next cells (parse pages, build indexes, P2 baselines).